In [1]:
# Improved Text+Video+Context GATv2 training
# (with fixes + accuracy tweaks)

# ===============================
# 0. Dependencies & Setup
# ===============================
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__)").html
!pip install -q torch_geometric matplotlib tqdm scikit-learn

import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GATv2Conv, global_mean_pool

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score

import matplotlib.pyplot as plt
from tqdm import tqdm

# --- CONFIGURATION ---
BASE_PATH = '/content/drive/MyDrive/Honors/'  # UPDATE THIS IF NEEDED
TEXT_FEAT = 'text_features_imp.npy'
VIDEO_FEAT = 'video_features_imp.npy'
CONTEXT_FEAT = 'context_features_imp.npy'
LABELS_FEAT = 'labels_imp.npy'

BATCH_SIZE = 16
EPOCHS = 150
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 30

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


ERROR: Could not find a version that satisfies the requirement pyg_lib (from versions: none)
ERROR: No matching distribution found for pyg_lib
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.0 MB/s eta 0:00:00
Using device: cuda


In [2]:
# ===============================
# 1. Load Features (Ablated: No Audio)
# ===============================
from google.colab import drive
drive.mount('/content/drive')

print("Loading Features (Text, Video, Context)...")

t_features = np.load(os.path.join(BASE_PATH, TEXT_FEAT))
v_features = np.load(os.path.join(BASE_PATH, VIDEO_FEAT))
c_features = np.load(os.path.join(BASE_PATH, CONTEXT_FEAT))
labels = np.load(os.path.join(BASE_PATH, LABELS_FEAT))

print(f"Loaded {len(labels)} samples.")
print(f"Dims: T={t_features.shape[1]}, V={v_features.shape[1]}, C={c_features.shape[1]}")

# class imbalance handling for BCEWithLogitsLoss
pos_count = labels.sum()
neg_count = len(labels) - pos_count
pos_weight_value = neg_count / max(pos_count, 1)
print("pos_weight for BCE:", pos_weight_value)


Mounted at /content/drive
Loading Features (Text, Video, Context)...
Loaded 690 samples.
Dims: T=768, V=2048, C=2816
pos_weight for BCE: 1.0


In [3]:
# ===============================
# 2. Build Graph Dataset (3 Nodes)
# ===============================

data_list = []
for i in range(len(labels)):
    t = torch.tensor(t_features[i], dtype=torch.float).unsqueeze(0)
    v = torch.tensor(v_features[i], dtype=torch.float).unsqueeze(0)
    c = torch.tensor(c_features[i], dtype=torch.float).unsqueeze(0)
    y = torch.tensor([labels[i]], dtype=torch.float)

    # We will ignore per-sample edge_index and instead use a fixed 3-node template
    data = Data(y=y)
    data.text = t
    data.video = v
    data.context = c
    data.num_nodes = 3  # 3 nodes: 0=T, 1=V, 2=C

    data_list.append(data)

print(f"Built {len(data_list)} graph samples (3 nodes each).")


Built 690 graph samples (3 nodes each).


In [4]:
# ===============================
# 3. Train/Val/Test Split
# ===============================

train_val, test = train_test_split(
    data_list, test_size=0.15, random_state=42, stratify=labels
)

labels_train_val = np.array([d.y.item() for d in train_val])
train, val = train_test_split(
    train_val, test_size=0.1765, random_state=42, stratify=labels_train_val
)  # ~70/15/15

print(f"Train: {len(train)}, Val: {len(val)}, Test: {len(test)}")

train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False)


Train: 482, Val: 104, Test: 104


In [5]:
# ===============================
# 4. SarcasmGATv2 Model (No Audio, fixed edges)
# ===============================

class SarcasmGATv2_NoAudio(nn.Module):
    def __init__(self, t_dim, v_dim, c_dim,
                 hidden_dim=256, gat_dropout=0.4, fc_dropout=0.4,
                 use_context_only=False):
        super().__init__()

        self.use_context_only = use_context_only

        # Projections
        self.p_t = nn.Sequential(
            nn.Linear(t_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.p_v = nn.Sequential(
            nn.Linear(v_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )
        self.p_c = nn.Sequential(
            nn.Linear(c_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
        )

        # GATv2 layers
        self.gat1 = GATv2Conv(hidden_dim, hidden_dim, heads=8,
                              concat=True, dropout=gat_dropout)
        self.gat2 = GATv2Conv(hidden_dim * 8, hidden_dim, heads=1,
                              concat=False, dropout=gat_dropout)

        # Classifier
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Dropout(fc_dropout),
            nn.Linear(128, 1),
        )

        # fixed 3-node edge template on device later
        self.register_buffer(
            "edge_template",
            torch.tensor([
                [0, 1], [1, 0],
                [2, 0], [0, 2], [2, 1], [1, 2],
                [0, 0], [1, 1], [2, 2],
            ], dtype=torch.long).t()  # (2, 9)
        )

    def forward(self, data):
        x_t = self.p_t(data.text)
        x_v = self.p_v(data.video)
        x_c = self.p_c(data.context)

        batch_size = x_t.size(0)
        d = x_t.size(1)

        # build node feature matrix for 3 nodes per graph
        x = torch.zeros(batch_size * 3, d, device=x_t.device)
        x[0::3] = x_t
        x[1::3] = x_v
        x[2::3] = x_c

        # replicate fixed edge template across batch
        edge_index_list = []
        for i in range(batch_size):
            edge_index_list.append(self.edge_template + i * 3)
        edge_index = torch.cat(edge_index_list, dim=1)

        out = self.gat1(x, edge_index)
        out = F.gelu(out)
        out = self.gat2(out, edge_index)

        # batch vector for pooling: 3 nodes per graph
        batch_vec = torch.arange(batch_size, device=x_t.device).view(-1, 1).repeat(1, 3).view(-1)

        if self.use_context_only:
            # reshape and take node 2 (context) per graph
            out_reshaped = out.view(batch_size, 3, -1)
            out_ctx = out_reshaped[:, 2, :]
            graph_emb = out_ctx
        else:
            graph_emb = global_mean_pool(out, batch_vec)

        logits = self.fc(graph_emb)
        return logits.squeeze(-1)


In [6]:
# ===============================
# 5. Optimizer, Scheduler, Loss
# ===============================

model = SarcasmGATv2_NoAudio(
    t_dim=t_features.shape[1],
    v_dim=v_features.shape[1],
    c_dim=c_features.shape[1],
    hidden_dim=256,
    gat_dropout=0.4,
    fc_dropout=0.4,
    use_context_only=False,  # toggle to True to pool only context node
).to(DEVICE)

# separate weight decay for some params
decay_params, no_decay_params = [], []
for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if any(nd in name for nd in ["bias", "LayerNorm"]):
        no_decay_params.append(param)
    else:
        decay_params.append(param)

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LEARNING_RATE,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, T_0=30, T_mult=2
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_value], device=DEVICE)
)


In [7]:
# ===============================
# 6. Train / Eval Functions
# ===============================

def train_one_epoch(loader):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for data in loader:
        data = data.to(DEVICE)
        optimizer.zero_grad()

        logits = model(data)
        loss = criterion(logits, data.y.view(-1))

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * data.y.size(0)

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        labels = data.y.view(-1).cpu().numpy().astype(int)

        all_preds.extend(preds)
        all_labels.extend(labels)

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    return avg_loss, acc, f1


@torch.no_grad()
def evaluate(loader, threshold=0.5):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []

    for data in loader:
        data = data.to(DEVICE)
        logits = model(data)

        probs = torch.sigmoid(logits).cpu().numpy()
        preds = (probs >= threshold).astype(int)
        labels = data.y.view(-1).cpu().numpy().astype(int)

        all_probs.extend(probs.tolist())
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds, average='macro')
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = float('nan')

    return acc, f1, prec, rec, auc, np.array(all_probs), np.array(all_labels)


@torch.no_grad()
def evaluate_raw(loader):
    model.eval()
    all_probs, all_labels = [], []
    for data in loader:
        data = data.to(DEVICE)
        logits = model(data)
        probs = torch.sigmoid(logits).cpu().numpy()
        labels = data.y.view(-1).cpu().numpy().astype(int)
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.tolist())
    return np.array(all_probs), np.array(all_labels)


In [8]:
# ===============================
# 7. Training Loop with Early Stopping
# ===============================

best_val_f1 = 0.0
best_state = None
patience_counter = 0

train_losses, val_f1s = [], []

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc, train_f1 = train_one_epoch(train_loader)
    val_acc, val_f1, val_prec, val_rec, val_auc, _, _ = evaluate(val_loader, threshold=0.5)

    scheduler.step(epoch)

    train_losses.append(train_loss)
    val_f1s.append(val_f1)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss {train_loss:.4f} Acc {train_acc:.4f} F1 {train_f1:.4f} | "
        f"Val Acc {val_acc:.4f} F1 {val_f1:.4f} Prec {val_prec:.4f} Rec {val_rec:.4f} AUC {val_auc:.4f} | "
        f"LR {optimizer.param_groups[0]['lr']:.6f}"
    )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("Early stopping triggered.")
        break

print(f"Training complete. Best Val F1: {best_val_f1:.4f}")

if best_state is not None:
    model.load_state_dict(best_state)
    torch.save(model.state_dict(), os.path.join(BASE_PATH, "best_sarcasm_gatv2_no_audio_improved.pth"))
    print("Best model saved.")


Epoch 001 | Train Loss 0.6965 Acc 0.4627 F1 0.4539 | Val Acc 0.5000 F1 0.3333 Prec 0.2500 Rec 0.5000 AUC 0.7374 | LR 0.000100
Epoch 002 | Train Loss 0.6928 Acc 0.5104 F1 0.5059 | Val Acc 0.5096 F1 0.3544 Prec 0.7524 Rec 0.5096 AUC 0.7888 | LR 0.000099
Epoch 003 | Train Loss 0.6853 Acc 0.5249 F1 0.5177 | Val Acc 0.5769 F1 0.5035 Prec 0.6884 Rec 0.5769 AUC 0.7833 | LR 0.000098
Epoch 004 | Train Loss 0.6410 Acc 0.6411 F1 0.6406 | Val Acc 0.5865 F1 0.5105 Prec 0.7287 Rec 0.5865 AUC 0.7792 | LR 0.000096
Epoch 005 | Train Loss 0.5921 Acc 0.6992 F1 0.6965 | Val Acc 0.7404 F1 0.7392 Prec 0.7448 Rec 0.7404 AUC 0.7903 | LR 0.000093
Epoch 006 | Train Loss 0.5905 Acc 0.7075 F1 0.7068 | Val Acc 0.7500 F1 0.7492 Prec 0.7534 Rec 0.7500 AUC 0.8018 | LR 0.000090
Epoch 007 | Train Loss 0.5490 Acc 0.7407 F1 0.7396 | Val Acc 0.7212 F1 0.7190 Prec 0.7280 Rec 0.7212 AUC 0.7837 | LR 0.000087
Epoch 008 | Train Loss 0.5155 Acc 0.7552 F1 0.7545 | Val Acc 0.6442 F1 0.6044 Prec 0.7415 Rec 0.6442 AUC 0.7862 | LR 0

In [9]:
# ===============================
# 8. Threshold tuning on Validation
# ===============================

val_probs, val_labels = evaluate_raw(val_loader)

best_f1, best_t = 0.0, 0.5
for t in np.linspace(0.2, 0.8, 25):
    preds = (val_probs >= t).astype(int)
    f1 = f1_score(val_labels, preds, average='macro')
    if f1 > best_f1:
        best_f1, best_t = f1, t

print("Best threshold on val:", best_t, "F1:", best_f1)


Best threshold on val: 0.8 F1: 0.6824280558897011


In [10]:
# ===============================
# 9. Final Test Evaluation (using best threshold)
# ===============================

# make sure model is in best_state already

# if you saved best_t manually from above cell, set it here
# best_t = 0.5  # replace with printed best threshold if needed

print("Using threshold:", best_t)

test_acc, test_f1, test_prec, test_rec, test_auc, _, _ = evaluate(test_loader, threshold=best_t)
print(f"TEST | Acc {test_acc:.4f} F1 {test_f1:.4f} Prec {test_prec:.4f} Rec {test_rec:.4f} AUC {test_auc:.4f}")


Using threshold: 0.8
TEST | Acc 0.7308 F1 0.7304 Prec 0.7321 Rec 0.7308 AUC 0.7844
